# qLDPC decoder methods

This notebook targets **CUDA-Q QEC 0.7** and follows the [official decoder example](https://nvidia.github.io/cudaqx/examples_rst/qec/decoders.html) and [`nv-qldpc-decoder` API](https://nvidia.github.io/cudaqx/api/qec/python_api.html#cudaq-qec-nv-qldpc-decoder). The bivariate-bicycle-code dataset contains a 216 x 865 sparse check matrix, 12 logical observables, and 10,000 labeled trials at the 0.01 noise setting.

A decoder takes a measured syndrome and uses the parity-check matrix to infer a likely error correction. Here, NVIDIA's GPU-accelerated QLDPC decoder performs iterative belief propagation (BP), then uses order-zero ordered-statistics decoding (OSD-0) as a second stage when BP does not converge. Every method uses all 10,000 shots: unbatched timing uses 10,000 `decode()` calls, while batched timing uses ten `decode_batch()` calls of 1,000 shots.

In [ ]:
import bz2
import json
import multiprocessing as mp
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from scipy.sparse import csr_matrix

qec_package_versions = []
for package in ("cudaq-qec-cu13", "cudaq-qec-cu12", "cudaq-qec"):
    try:
        qec_package_versions.append(version(package))
    except PackageNotFoundError:
        pass
qec_version = next((v for v in qec_package_versions if v.startswith("0.7.")), None)
if qec_version is None:
    raise RuntimeError(
        f"This notebook requires CUDA-Q QEC 0.7.x; found {qec_package_versions or 'no installation'}"
    )
print(f"CUDA-Q QEC: {qec_version}")

def parse_csr_mat(data, shape, name):
    indptr = np.asarray(data[f"{name}_indptr"], dtype=int)
    indices = np.asarray(data[f"{name}_indices"], dtype=int)
    values = np.ones(len(indices), dtype=np.uint8)
    return csr_matrix((values, indices, indptr), shape=shape)

def parse_H_csr(data, shape):
    return parse_csr_mat(data, shape, "H")

def parse_observables(data, shape):
    # CUDA-Q QEC 0.7 expects O to have shape (observables, error mechanisms).
    return parse_csr_mat(data, shape, "obs_mat").T.toarray(order="C")

filename = "osd_216_865_0.01.json"
data_path = Path(tempfile.gettempdir()) / filename
url = f"https://github.com/NVIDIA/cudaqx/releases/download/0.2.0/{filename}.bz2"
if not data_path.exists():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data_path.write_bytes(bz2.decompress(response.content))

data = json.loads(data_path.read_text())
H = parse_H_csr(data, data["shape"])
observable_matrix = parse_observables(data, data["obs_mat_shape"])
error_rates = np.asarray(data["error_rate_vec"])
trials = data["trials"]
syndromes = np.asarray([trial["syndrome_truth"] for trial in trials], dtype=np.uint8)
observable_truth = np.asarray([trial["obs_truth"] for trial in trials], dtype=np.uint8)
NUM_SHOTS = data["num_trials"]
BATCH_SIZE = 1_000
print(f"H: {H.shape}; observables: {observable_matrix.shape[0]}; shots: {NUM_SHOTS}")

## Shared decoder loop

The parsing, timing, and logical-error-rate (LER) calculation stay the same for every method; only `decoder_options` changes below. CUDA-Q QEC 0.7 accepts the sparse SciPy matrix `H` directly and, when given `O`, returns logical-observable flips instead of the full correction vector. The batched path uses the packed `BatchDecoderResult.result` array introduced in 0.7, avoiding per-shot result copies. Each method gets a fresh GPU process so full-dataset configurations release their GPU state before the next benchmark.

In [ ]:
benchmark_rows = []

def _decoder_worker(name, decoder_options, output):
    import cudaq_qec as qec
    if not qec.__version__.startswith("CUDA-Q QEC 0.7."):
        raise RuntimeError(f"Expected CUDA-Q QEC 0.7.x, found {qec.__version__}")

    common_options = {
        "max_iterations": 50,
        "error_rate_vec": error_rates,
        "O": observable_matrix,
        "use_sparsity": True,
        "use_osd": True,
        "osd_method": 1,  # OSD-0
        "osd_order": 0,
    }

    measurements = {}
    for mode in ("unbatched", "batched"):
        size = 1 if mode == "unbatched" else BATCH_SIZE
        decoder = qec.get_decoder(
            "nv-qldpc-decoder", H,
            **common_options, **decoder_options, bp_batch_size=size,
        )

        elapsed = 0.0
        logical_errors = 0
        if mode == "unbatched":
            for syndrome, actual in zip(syndromes, observable_truth):
                start = time.perf_counter()
                result = decoder.decode(syndrome)
                elapsed += time.perf_counter() - start
                predicted = np.asarray(result.result, dtype=np.uint8)
                logical_errors += int(np.any(predicted != actual))
        else:
            for first in range(0, NUM_SHOTS, BATCH_SIZE):
                last = min(first + BATCH_SIZE, NUM_SHOTS)
                start = time.perf_counter()
                results = decoder.decode_batch(syndromes[first:last])
                elapsed += time.perf_counter() - start
                predicted = np.asarray(results.result, dtype=np.uint8)
                logical_errors += np.count_nonzero(
                    np.any(predicted != observable_truth[first:last], axis=1)
                )

        measurements[mode] = {
            "ms_per_shot": 1e3 * elapsed / NUM_SHOTS,
            "errors": logical_errors,
        }

    unbatched = measurements["unbatched"]
    batched = measurements["batched"]
    output.put({
        "method": name, "shots": NUM_SHOTS, "batch size": BATCH_SIZE,
        "unbatched ms / shot": unbatched["ms_per_shot"],
        "batched ms / shot": batched["ms_per_shot"],
        "speedup": unbatched["ms_per_shot"] / batched["ms_per_shot"],
        "unbatched errors": unbatched["errors"],
        "batched errors": batched["errors"],
        "unbatched LER": unbatched["errors"] / NUM_SHOTS,
        "batched LER": batched["errors"] / NUM_SHOTS,
    })

def run_method(name, decoder_options):
    context = mp.get_context("fork")
    output = context.Queue()
    process = context.Process(
        target=_decoder_worker, args=(name, decoder_options, output)
    )
    process.start()
    process.join()
    if process.exitcode != 0:
        raise RuntimeError(f"{name} decoder process failed")
    row = output.get()
    output.close()
    benchmark_rows.append(row)
    display(pd.DataFrame([row]).style.format({
        "unbatched ms / shot": "{:.3f}",
        "batched ms / shot": "{:.3f}", "speedup": "{:.1f}x",
        "unbatched LER": "{:.2%}", "batched LER": "{:.2%}",
    }))
    return row

## Sum-product BP

Sum-product is the classic BP update rule (`bp_method=0`, the default). It passes probability-based messages between variable and check nodes in the sparse Tanner graph, making it the natural baseline for the other methods. This is still iterative BP rather than an exact maximum-likelihood decoder, so OSD-0 handles syndromes for which BP does not converge.

In [ ]:
sum_product = run_method(
    "Sum-product",
    {
        "bp_method": 0,  # Sum-product BP
    },
)

## Min-sum BP

Min-sum BP (`bp_method=1`) approximates the sum-product check-node update with minimum operations. That simpler update is often faster, but its approximation can change convergence and accuracy. `scale_factor` rescales the min-sum messages; the value `1.0` below leaves them unscaled.

In [ ]:
min_sum = run_method(
    "Min-sum",
    {
        "bp_method": 1,  # Min-sum BP
        "scale_factor": 1.0,  # Message scaling
    },
)

## Memory BP

Memory BP (`bp_method=2`) augments min-sum BP with the same memory strength at every variable node. The CUDA-Q docs describe this memory as a way to help BP escape local minima when standard BP fails to converge. It requires sparse processing, enabled in the shared options, and `gamma0` sets the uniform memory strength.

In [ ]:
memory_bp = run_method(
    "Memory BP",
    {
        "bp_method": 2,  # Uniform-memory min-sum
        "gamma0": 0.5,  # Uniform memory strength
    },
)

## Disordered-memory BP

Disordered-memory BP (`bp_method=3`) is the min-sum memory variant with a separate memory strength for each variable node, allowing the update to adapt to the code structure. CUDA-Q can accept explicit strengths or sample them from `gamma_dist`; this notebook samples from `[0.1, 0.5]`. `bp_seed` fixes that sampling so the decoder configuration is reproducible.

In [ ]:
dmem_bp = run_method(
    "DMem BP",
    {
        "bp_method": 3,  # Disordered-memory min-sum
        "gamma_dist": [0.1, 0.5],  # Memory-strength range
        "bp_seed": 42,  # Reproducible strengths
    },
)

## Sequential relay BP

Sequential Relay BP (`composition=1`) runs several disordered-memory BP legs in sequence, each with a different gamma configuration. The decoder first performs `pre_iter` warm-up iterations using `gamma0`, then tries `num_sets` relay legs; `FirstConv` stops as soon as one leg converges. This can improve difficult-syndrome convergence at the cost of extra work when multiple legs are needed.

In [ ]:
relay_bp = run_method(
    "Relay BP",
    {
        "bp_method": 3,  # Disordered-memory BP legs
        "composition": 1,  # Sequential relay composition
        "gamma0": 0.3,  # Pre-relay memory strength
        "gamma_dist": [0.1, 0.5],
        "bp_seed": 42,
        "srelay_config": {
            "pre_iter": 5,
            "num_sets": 3,
            "stopping_criterion": "FirstConv",
        },
    },
)

## Comparison

This notebook keeps CUDA-Q's default `repeatable=False`, so results are not guaranteed to be bit-for-bit repeatable; in the saved run, the Relay BP counts differ by one shot between the two execution modes. The 0.7 API provides `repeatable=True` for repeatable results, but it also requires a nonzero `clip_value` and carries a documented performance cost, so this timing comparison retains the default behavior.

In [ ]:
pd.DataFrame(benchmark_rows).style.format({
    "unbatched ms / shot": "{:.3f}",
    "batched ms / shot": "{:.3f}", "speedup": "{:.1f}x",
    "unbatched LER": "{:.2%}", "batched LER": "{:.2%}",
})